# Monte Carlo Simulation for European and Binary Option Pricing

## CQF Exam 2

**Author:** Mao Yikai  
**Date:** 2026/4/25  
**Objective:** Implement and compare numerical schemes for derivative pricing using risk-neutral valuation

---

## 1. Introduction (25%)

### 1.1 Problem Statement

The pricing of derivative securities is a fundamental problem in quantitative finance. Under the risk-neutral measure Q, the value of a derivative with payoff $\Phi(S_T)$ expiring at time $T$ is given by:

$$V(S, t) = e^{-r(T-t)} \mathbb{E}^Q[\Phi(S_T)]$$

where:
- $S$ is the underlying asset price
- $r$ is the constant risk-free interest rate
- $\mathbb{E}^Q[\cdot]$ denotes expectation under the risk-neutral measure

For derivatives without closed-form solutions, Monte Carlo simulation is a useful approach. In this project, we implement 3 numerical schemes to simulate the underlying asset price dynamics, evaluate their accuracy, and compare variance reduction techniques.

### 1.2 Geometric Brownian Motion and Black-Scholes Framework

Under the risk-neutral measure, the stock price evolves according to:

$$dS_t = rS_t dt + \sigma S_t dW_t$$

where $\sigma$ is the volatility and $W_t$ is a standard Brownian motion. The solution to this SDE is:

$$S_T = S_0 \exp\left(\left(r - \frac{\sigma^2}{2}\right)(T-t) + \sigma \sqrt{T-t} Z\right)$$

where $Z \sim \mathcal{N}(0,1)$ is a standard normal random variable. For European call options, the closed-form Black-Scholes formula provides:

$$C = S_0 N(d_1) - E e^{-r(T-t)} N(d_2)$$

where $d_1 = \frac{\ln(S_0/E) + (r + \sigma^2/2)(T-t)}{\sigma\sqrt{T-t}}$ and $d_2 = d_1 - \sigma\sqrt{T-t}$, with $N(\cdot)$ being the cumulative standard normal distribution (Hull, 2018).

### 1.3 Numerical Discretization Schemes

To compute the expectation numerically, we discretize the SDE over a time grid $0 = t_0 < t_1 < \ldots < t_N = T$ with step size $\Delta t = T/N$.

#### 1.3.1 Euler-Maruyama Scheme

The Euler-Maruyama scheme (Maruyama, 1955) provides the simplest approximation by discretizing the drift and diffusion terms:

$$S_{n+1} = S_n + rS_n \Delta t + \sigma S_n \Delta W_n$$

where $\Delta W_n \sim \mathcal{N}(0, \Delta t)$.

#### 1.3.2 Milstein Scheme

The Milstein scheme (Milstein, 1974) improves upon Euler-Maruyama by adding a correction term:

$$S_{n+1} = S_n + rS_n \Delta t + \sigma S_n \Delta W_n + \frac{1}{2}\sigma^2 S_n ((\Delta W_n)^2 - \Delta t)$$

This extra term captures nonlinear effects in the volatility dynamics, providing better accuracy for strong convergence in path-dependent scenarios.

#### 1.3.3 Closed Form Solution

We use the exact solution from section 1.2 to generate reference option prices via Monte Carlo. This provides a benchmark to assess the accuracy of the discretized schemes.

#### 1.3.4 Convergence Classification

The 3 schemes exhibit different convergence behaviors. We distinguish between **weak** and **strong** convergence:

- **Weak Convergence:** Convergence of expectations, $E[S_n^{\text{scheme}}] \to E[S_n^{\text{exact}}]$, relevant for option pricing. 
  - Euler-Maruyama: $O(\Delta t)$ weak order
  - Milstein: $O(\Delta t^2)$ weak order
  
- **Strong Convergence:** Convergence of sample paths, $E[|S_n^{\text{scheme}} - S_n^{\text{exact}}|] \to 0$, relevant for path-dependent options.
  - Euler-Maruyama: $O(\Delta t^{0.5})$ strong order
  - Milstein: $O(\Delta t^{1.0})$ strong order

For European options, where only the terminal payoff matters, weak convergence dominates. In practice, Monte Carlo variance ($O(N^{-0.5})$) typically dominates weak discretization bias, making higher-order weak schemes less critical than sample size (Kloeden & Platen, 1992).

We will apply these schemes to price 2 types of options with different payoff structures:

### 1.4 Option Types and Payoffs

**European Call Option:**  
$$\Phi_{EU}(S_T) = \max(S_T - E, 0)$$

**Binary Call Option:**  
$$\Phi_{Binary}(S_T) = \mathbb{1}_{S_T > E}$$

### 1.5 Variance Reduction: Antithetic Variates

The antithetic variates technique (Hammersley & Handscomb, 1964) reduces Monte Carlo variance through symmetry. For each path $Z$, we generate a paired path $-Z$ and average their payoffs:

$$\hat{V}_{AV} = \frac{1}{2M} \sum_{i=1}^{M} \left[ e^{-rT} \Phi(S^{(i)}_T) + e^{-rT} \Phi(S^{(-i)}_T) \right]$$

This typically reduces variance by 50-90% for smooth payoffs, improving estimation accuracy without additional computational cost proportional to simulation count (Glasserman, 2004).